# Exp5 Part A — Dry run + full test blocking

**Goal:** prove the heavy step works (dry run), then generate ALL test candidates.

**You run:** setup cell → dry-run cell (paste output, WAIT for go) → full-blocking cell → union cell.
Then Part B (separate notebook) does train + inference + submission.

**If the runtime dies:** re-run from the top — `git lfs pull` and every `--resume`
cell skip finished work. Files already copied to Drive survive.

**Locked config:** char (3,4), min_df=2, max_df=0.05, K=40/channel.
Countries are derived from `test_source1.tsv`, never hardcoded.

In [ ]:
# 0) Setup: Drive, repo (clone once, pull after), data, deps. (~15-25 min first time)
BRANCH = "arena/01a0e27b-ml-challenge"
from google.colab import drive
drive.mount('/content/drive')
!apt-get -qq update && apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install > /dev/null 2>&1
!if [ -d /content/ML-challenge ]; then cd /content/ML-challenge && git checkout {BRANCH} && git pull --ff-only; else git clone -b {BRANCH} https://github.com/Saadmadni84/ML-challenge.git /content/ML-challenge; fi
%cd /content/ML-challenge
!git lfs pull
!wc -l dataset/train/*.tsv dataset/test/*.tsv
!pip -q install -r code/business_entity_resolution/requirements.txt
!python3 -c "import sklearn, lightgbm, rapidfuzz, sparse_dot_topn; print('deps OK')"
# Countries are DERIVED from the data (open set - never hardcoded).
import csv, os
from collections import Counter
os.makedirs("output", exist_ok=True)
c = Counter()
with open("dataset/test/test_source1.tsv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        c[(row.get("country") or "").strip()] += 1
COUNTRIES = sorted(c)
print("test S1 countries:", dict(c))
assert all(COUNTRIES), "blank country label found in test_source1.tsv!"


In [ ]:
# 1) DRY RUN: 3k test queries/country against FULL targets (~20-40 min).
# Builds the full indices (the heaviest step) - that is the point. PASTE OUTPUT.
# (subprocess, not `!`, so the literal {country} placeholder reaches blocking.py.)
import subprocess
S = "code/business_entity_resolution/src"
for field in ["name", "address"]:
    cmd = ["python3", f"{S}/blocking.py",
           "--s1", "dataset/test/test_source1.tsv",
           "--s2", "dataset/test/test_source2.tsv",
           "--s3", "dataset/test/test_source3.tsv",
           "--field", field, "--out", f"output/dry_{field}_{{country}}.tsv",
           "--k", "40", "--min-ngram", "3", "--max-ngram", "4",
           "--min-df", "2", "--max-df", "0.05",
           "--countries", ",".join(COUNTRIES), "--max-q", "3000"]
    print("$ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


⏸️ **PAUSE — paste the dry-run output (index sizes, S1/s, ETAs) in chat and WAIT.**
We confirm the budget before launching the hours-long full-blocking cell.

In [ ]:
# 2) FULL TEST BLOCKING, per country+field with --resume (hours; ETA printed).
# Run ONLY after the dry run was approved. Leave the tab open. Re-runnable:
# finished shards are skipped, and Drive copies survive a dead runtime.
import os, shutil, subprocess
S = "code/business_entity_resolution/src"
for C in COUNTRIES:
    for F in ["name", "address"]:
        out = f"output/test_{C}_{F}.tsv"
        cmd = ["python3", f"{S}/blocking.py",
               "--s1", "dataset/test/test_source1.tsv",
               "--s2", "dataset/test/test_source2.tsv",
               "--s3", "dataset/test/test_source3.tsv",
               "--field", F, "--out", f"output/test_{{country}}_{F}.tsv",
               "--k", "40", "--min-ngram", "3", "--max-ngram", "4",
               "--min-df", "2", "--max-df", "0.05",
               "--countries", C, "--resume"]
        print(f"=== {C} / {F} ===", flush=True)
        subprocess.run(cmd, check=True)
        assert os.path.getsize(out) > 100, f"suspiciously small: {out}"
        shutil.copy(out, "/content/drive/MyDrive/" + os.path.basename(out))
        print(f"backed up {out}", flush=True)
!ls -lh output/test_*.tsv


In [ ]:
# 3) Test unions (streaming, low RAM) + backup to Drive. End of Part A.
import os, shutil, subprocess
S = "code/business_entity_resolution/src"
for C in COUNTRIES:
    out = f"output/test_union_{C}.tsv"
    subprocess.run(["python3", f"{S}/union_candidates.py",
                    "--a", f"output/test_{C}_name.tsv",
                    "--b", f"output/test_{C}_addr.tsv",
                    "--k-per-file", "40", "--out", out], check=True)
    shutil.copy(out, "/content/drive/MyDrive/" + os.path.basename(out))
!ls -lh output/test_*.tsv


## Part A done — send back

1. Dry-run output (cell 1) — sent before the full-blocking cell.
2. `ls -lh` of `output/test_*.tsv` (tail of the last two cells).

Then open **Part B**: `02b_colab_train_and_submit.ipynb` (train + inference +
submission). It restores Part A files from Drive if the runtime is fresh.